# Regression

**Working Example.** Copy this file, rename it, and modify *your* copy.
See earlier notebooks for more information. 

- Author: Denise Case
- Date: 2026-06
- Dataset: Seaborn Penguins
- Target: body_mass_g

Run all cells top to bottom (**Run All**) before pushing to GitHub.

## M4. Regression

This notebook fits a line and computes the numbers analysts use to judge a fit:

- R-squared
- RMSE
- residuals
  
It shows how complexity changes train vs. test error. 

The analyst decides whether the fit is "good" and 
how much complexity is right. 
Code calculates values, but the analyst decides.

Every time we fit a model, we use the same shape: 
setup, load, prepare, model, evaluate, summarize.

Judgment always remains with the analyst.

## Overview

This project uses the penguins dataset.
We choose to predict the target `body_mass_g`.
This target is a **numeric** variable (rather than a discrete category), so we have a:

- supervised ML problem (because we've chosen a target)
- a regression problem (because our target is numeric)

Customize the overview in your copy to reflect your dataset and choices.

## A. Prepare the Project Environment (.venv/)

- Open one project in VS Code at a time.
- Prepare the .venv/: specify Python version and install / upgrade dependencies listed in `pyproject.toml`.
- Open an integrated terminal (PowerShell if Windows) in the **root project** folder and run:

```shell
uv self update
uv python pin 3.14
uv lock --upgrade
uv sync --extra dev --extra docs --upgrade
```

## B. Select the Notebook Kernel

- Click on the **Select Kernel** name in the top-right corner of the notebook interface.
- Choose Python Environments... /
- Choose the recommended local .venv/ from the drop-down menu.
- This will create a new kernel for the notebook and allow the notebook to use packages installed in the .venv/ environment.

## C. Working in Notebooks (Custom Notes)

- To run a cell, press **Ctrl+Enter** (or **Cmd+Enter** on Mac) when done editing the cell.
- Change the type of a cell (e.g., code or markdown) by looking in the lower left corner of the notebook interface.
- Rearrange cells by dragging and dropping them within the notebook.

See [Run Jupyter Notebooks](https://denisecase.github.io/pro-analytics-02/workflow-b-apply-example-project/run-notebook/) for:

- how to **copy a notebook**
- how to release a `project.log` file
- how to deal with a **stuck kernel**
- etc.

## Section 0. When to Use Regression, and How We Judge It

Use **regression** when the target is a **number** - mass, price, temperature,
demand. (A category target is classification, Module 3.)

We judge a regression on held-out data, with:

- **R-squared** - the fraction of the variation in the target the model explains
  (roughly 0 to 1; higher is more).
- **RMSE** - root mean squared error, the typical size of a miss, in the target's units.
- **residual plot** - actual minus predicted. If a straight line fits well, residuals
  scatter randomly around zero with no pattern. A curve or a funnel says the line is
  the wrong shape - itself a useful finding.

**Over- vs. under-fitting:** a model too simple **underfits** (misses real structure;
poor on both train and test). A model too complex **overfits** (memorizes the training
data; great on train, worse on test). You detect it by comparing train and test error
as complexity changes - Section 6.

## Section 1. Project Setup and Imports

In [ ]:
# === Section 1a. DECLARE IMPORTS ===

from importlib.metadata import version  # to verify
import logging  # for type hinting
import platform  # to verify
from typing import Final  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("M04", level="DEBUG")
log_header(LOG, "M04")

# === Section 1c. USE THE LOGGER TO VERIFY IMPORTS ===

# If any do NOT return a version number, then that package is not installed correctly.
# Check your pyproject.toml and re-run environment setup commands.

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")
LOG.info(f"  numpy:        {version('numpy')}")
LOG.info(f"  scikit-learn: {version('scikit-learn')}")
LOG.info(f"  seaborn:      {version('seaborn')}")
LOG.info(f"  matplotlib:   {version('matplotlib')}")

# === Section 1d. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## Section 2. Load and Prepare the Data

In [ ]:
# === Section 2. Load and Prepare the Data ===

# We are loading data from Seaborn's built-in datasets,
# which are small datasets included with the Seaborn library
# for practice and demonstration purposes.
# They are listed above with sns.get_dataset_names().
# Experiment with different ones to see what they contain.
# You can also load from CSV files, databases, or APIs and process is similar.

# CUSTOM: In this example, I load a Seaborn dataset by name (no external CSV).
# Change to explore a different dataset.
DATASET_NAME: Final[str] = "penguins"

LOG.info(f"Loading dataset: {DATASET_NAME}")
df: pd.DataFrame = sns.load_dataset(DATASET_NAME)
LOG.info(f"Loaded: {df.shape[0]} rows (instances), {df.shape[1]} columns")


# CUSTOM: ANALYST CHOICE - one numeric feature (x) and one numeric target (y).
# This pairing is expected to be close to a straight line; we still fit and look.
FEATURE_COL: Final[str] = "flipper_length_mm"
TARGET_COL: Final[str] = "body_mass_g"

FEATURE_LABEL: Final[str] = "Flipper length (mm)"
TARGET_LABEL: Final[str] = "Body mass (g)"

df_model: pd.DataFrame = df.dropna(subset=[FEATURE_COL, TARGET_COL]).copy()
LOG.info(f"Original rows: {df.shape[0]}  Model rows: {df_model.shape[0]}")

## Section 3. Split into Train and Test


ANALYST CHOICE

We hold out a test set the model never sees
so R-squared and RMSE reflect new data. 

Without a held-out set we cannot tell a good fit from memorization.

In [ ]:
# === Section 3. Split into Train and Test ===

# CUSTOM: ANALYST CHOICE

# Reproducibility for the split and the model.
# Pick any integer, but keep it the same to get the same results.
RANDOM_STATE: Final[int] = 42

"""Build X (2-D) and y (1-D), then split into train and test.

WHY: scikit-learn wants X as shape (n, 1) and y as shape (n,).

Double brackets give a DataFrame (2-D);
single brackets give a Series (1-D).
"""

# Declare X to hold the feature values,
# which will be used for model fitting.
X: np.ndarray = df_model[[FEATURE_COL]].to_numpy()

# Declare y to hold the target values,
# which will be used for model fitting.
y: np.ndarray = df_model[TARGET_COL].to_numpy()

# Declare X_train to hold the training feature values,
# which will be used for model fitting.
X_train: np.ndarray

# Declare X_test to hold the test feature values,
# which will be used for evaluation after model training.
X_test: np.ndarray

# Declare y_train to hold the training target values,
# which will be used for model fitting.
y_train: np.ndarray

# Declare y_test to hold the test target values,
# which will be used for evaluation after model training.
y_test: np.ndarray

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
LOG.info(f"Train instances: {len(X_train)}")
LOG.info(f"Test instances: {len(X_test)}")

## Section 4. Fit a Straight Line

In [ ]:
# === Section 4. Fit a Straight Line ===


"""
Fit y = slope * x + intercept with scikit-learn.

WHY: create -> .fit() -> read results / .predict()
 is the same pattern for every scikit-learn model.

The model exposes .coef_ (slope) and .intercept_.
"""

LOG.info("Fitting a linear regression (scikit-learn)")

# create a linear regression model object (untrained)
model = LinearRegression()

# fit the model to the training data (learn the slope and intercept)
model.fit(X_train, y_train)

# retrieve the results (slope and intercept) and log them

slope: float = float(model.coef_[0])
intercept: float = float(model.intercept_)

LOG.info(f"Fitted line: {TARGET_COL} = {slope:.6g} * {FEATURE_COL} + {intercept:.6g}")

## Section 5. Evaluate the Model Fit

In [ ]:
# === Section 5. Evaluate the Model Fit ===

# With regression, we use  R-squared, RMSE, and Residuals

"""
Report test R-squared and RMSE, and show a residual plot.

WHY: These are how the analyst decides
whether the line is a fair description.

This reports them on held-out data;
it does NOT declare the fit good or bad.

The analyst uses the numbers and the residual plot to decide.
"""

y_hat: np.ndarray = model.predict(X_test)

# Use float() to convert from numpy types to native Python types
residuals: np.ndarray = y_test - y_hat
r_squared: float = float(model.score(X_test, y_test))
rmse: float = float(np.sqrt(np.mean(residuals**2)))

LOG.info("Fit numbers on the TEST set (for you to interpret):")
LOG.info(f"  R-squared: {r_squared:.4f}")
LOG.info(f"  RMSE:      {rmse:.6g}  (in units of {TARGET_LABEL})")

# Residual plot: random scatter around 0 means a straight line fits well.

# Start a new figure
plt.figure()

# Create a scatter plot of residuals vs. predicted values
sns.scatterplot(x=y_hat, y=residuals)

# Add a horizontal line at y=0 to help visualize the residuals
plt.axhline(0)

# Label the axes and add a title
plt.xlabel(f"Predicted {TARGET_LABEL}")
plt.ylabel(f"Residual ({TARGET_LABEL})")
plt.title("Residuals vs. predicted (test set)")

# show the plot
plt.show()

### Interpretation

If residuals show a curve or a funnel, 
a straight line is the wrong shape. 

Provide your interpretation in Markdown. 

Finding out a relationship is not linear is still a valuable finding.

In this case, the residuals are pretty random about 0, with no obvious curve or funnel, 
so it looks like a straight line (degree=1) provides a good model. 

## Section 6. Detect Over- and Under-Fitting

Fit polynomials of increasing **degree** 
(degree 1 is the straight line) and plot
train vs. test RMSE. 

Underfitting shows as both errors high; 
overfitting shows as train RMSE falling while test RMSE rises. 

The best degree is where **test error is lowest** 
and the train/test gap is small.

The analyst decides which degree that is.

In [ ]:
# === Section 6. Detect Over- and Under-Fitting ===

"""Show train vs. test RMSE as polynomial degree increases.

WHY: Error is not a fixed property of "the model"; it moves with complexity.
Watching train and test diverge is how you detect over- and under-fitting and
pick complexity on purpose. numpy fits the polynomial; nothing hidden.
"""

# Call .ravel() to convert from
# shape (n, 1) to shape (n,) for np.polyfit and np.polyval.
x_train: np.ndarray = X_train.ravel()
x_test: np.ndarray = X_test.ravel()

# CUSTOM: ANALYST CHOICE
# for penguins, sweep from 1 to 10 as a good range.
# THIS IS DATA SET DEPENDENT and ANALYST CHOICE.

MAX_SWEEP: Final[int] = 10

# Declare lists to hold the degrees and corresponding train/test RMSE.
degrees: list[int] = list(range(1, MAX_SWEEP + 1))
train_rmse: list[float] = []
test_rmse: list[float] = []

for degree in degrees:
    # Calculate the coefficients of the polynomial fit for the given degree
    coeffs: np.ndarray = np.polyfit(x_train, y_train, degree)

    # Calculate the predicted values for the training and test sets using the polynomial coefficients
    train_pred: np.ndarray = np.polyval(coeffs, x_train)
    test_pred: np.ndarray = np.polyval(coeffs, x_test)

    # Calculate and store the RMSE for the training and test sets
    train_rmse.append(float(np.sqrt(np.mean((y_train - train_pred) ** 2))))
    test_rmse.append(float(np.sqrt(np.mean((y_test - test_pred) ** 2))))
    LOG.debug(
        f"  degree={degree:2d}  train RMSE={train_rmse[-1]:.6g}  test RMSE={test_rmse[-1]:.6g}"
    )

# start a new figure
plt.figure()

# Plot the train and test RMSE against the polynomial degree
# to visualize overfitting and underfitting.
plt.plot(degrees, train_rmse, marker="o", label="train RMSE")
plt.plot(degrees, test_rmse, marker="s", label="test RMSE")

# Label the axes and add a title
plt.xlabel("polynomial degree (model complexity)")
plt.ylabel(f"RMSE ({TARGET_LABEL})")
plt.title("Train vs. test RMSE as complexity increases")

# Add a legend
plt.legend()

# Show the plot
plt.show()

### Interpretation

If you see: 

- Both errors high at low degree = underfitting.
- Train RMSE falling while test RMSE rises = overfitting.

Pick a degree and defend it.

## Section 7. Summary and Next Steps

First, output key information (may use Python)
Second, provide your narrative, conclusions, and next steps (in Markdown)

In [ ]:
# === Python Summary ===


"""Record what was fit and the judgment still owed."""
slope: float = float(model.coef_[0])
intercept: float = float(model.intercept_)
LOG.info("========================")
LOG.info("SUMMARY")
LOG.info("========================")
LOG.info(f"Dataset:  {DATASET_NAME}")
LOG.info(f"Feature:  {FEATURE_COL}  ->  Target: {TARGET_COL}  (regression)")
LOG.info(f"Line:     {TARGET_COL} = {slope:.6g} * {FEATURE_COL} + {intercept:.6g}")
LOG.info("========================")


### Custom Narrative

Summarize your work in this Markdown cell in your notebook.

YOUR JUDGMENT (include a summary in docs/index.md also):

- Is a straight line a fair description? What did the residuals show?
- What is the RMSE in real units, and is that accurate enough?
- What degree did you choose, and why?

I might answer: 

Test RMSE is lowest at degree 1 (380.695) and rises from degree 2 onward. 
Train RMSE keeps falling, but test never improves - it's already overfitting at degree 2.

The RankWarning at degree 10 confirms it: 
the polynomial is becoming numerically unstable at high degrees, 
which is another sign we're well past the useful range.

When test RMSE is lowest at degree 1 and adding complexity only makes test worse, 
the simplest model wins. 
A straight line fits this data better than any curve.
That is also supported by the residuals not showing any kind of a funnel or curve.

There appears to be a linear relationship (degree = 1) between 
Feature: flipper_length_mm and Target: body_mass_g.

### Next Steps

Summarize your next steps in this Markdown cell in your notebook.


## Task: Make the Notebook Yours (Apply / Extend / Explore)

This is an example.
Copy this notebook and make it your own. 

In your copy:

1. At the beginning, update the Author, the purpose, the target, etc. 
2. Remove any instructions you do not need. 

Try things like the following.

1. **Apply** - Change `FEATURE_COL`/`TARGET_COL` 
   to a pairing you expect to be
   messier (e.g., a different measurement), and 
   compare the residual plot to this clean one. 
   A messier residual pattern is a real result.
2. **Extend** - Add a second feature and 
   fit a multiple regression (`df[[col_a, col_b]]`).
   Report whether the second feature improves test RMSE.
3. **Explore** - Find a degree that clearly overfits, and 
   explain in `docs/index.md` how you knew 
   from the train/test curves, not from fit alone.

## Task: Finalize Your README.md

Include in README.md:

- your project description
- any instructions
- your commands
- a link to key artifacts (including your executed notebook)

When done, you may delete this instruction in your custom notebook.


## Task: Finalize your docs/index.md

In your docs/index.md, include things like:

- Your target.
- Why predicting this target could be useful and for whom.
- Whether ML is a good tool for this problem and why or why not. 
- For example, would a simple rule work better?
- Which features look informative, and which look irrelevant.
- What might 'good enough' mean for this question.

Important:

- There is no threshold that answers these questions automatically.
- Use notebook, README.md, and docs/index.md to share your judgement and your ML skills.


## Task: Final Check

- `README.md` - reflects your description, instructions, commands, and links to your executed notebook.
- `docs/index.md` - reflects your project-specific updates.
- Your GitHub **About** section has a link to your hosted documentation site.
- The executed example notebook AND your custom notebook are available in `notebooks/`.
- Keep this **working example** alongside your custom work until your work has been assessed.
- Ensure your **custom notebook** introduces and narrates **your** custom project.

## Reminder: Run All before pushing to GitHub

Before saving a notebook (and running git add-commit-push), click 'Run All' to generate all outputs and display them in the notebook.

Follow our [pro-analytics-02](https://denisecase.github.io/pro-analytics-02/) common workflows.

Your README.md should have a description, a link to your executed notebook, and a list of commands (updated as you add your custom description, instructions, and commands).

Your docs/ folder should document your custom project analysis in the `docs/index.md` summary.
